In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
os.makedirs("charts", exist_ok=True)

# ------------------------------------------------------------------
# STEP 1: Sawal pehle puchho (Task ka pehla point)
# ------------------------------------------------------------------
# Q1. Kya aurat hone se survive karne ke chance badhe?
# Q2. Kya passenger class (1st/2nd/3rd) ka survival par asar tha?
# Q3. Kya umar (age) ka survival se koi relation hai?
# Q4. Kya zyada kiraya (fare) dene walon ke bachne ke chance zyada the?

# ------------------------------------------------------------------
# STEP 2: Data load karo aur structure dekho
# ------------------------------------------------------------------
df = sns.load_dataset("titanic")   # internet chahiye. Offline ho to: pd.read_csv("titanic.csv")

print("Shape (rows, columns):", df.shape)
print("\nFirst 5 rows:\n", df.head())
print("\nData types & non-null counts:")
df.info()
print("\nStatistical summary:\n", df.describe())

# ------------------------------------------------------------------
# STEP 3: Data issues dhundho (missing values, duplicates)
# ------------------------------------------------------------------
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Cleaning:
# - 'deck' mein bahut zyada missing hai, isliye hata do
# - 'age' ko median se fill karo
# - 'embarked' / 'embark_town' ko mode se fill karo
df = df.drop(columns=["deck"])
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])

print("\nCleaning ke baad missing values:", df.isnull().sum().sum())

# ------------------------------------------------------------------
# STEP 3.5: Outliers (IQR method se) fare mein
# ------------------------------------------------------------------
q1, q3 = df["fare"].quantile([0.25, 0.75])
iqr = q3 - q1
outliers = df[(df["fare"] < q1 - 1.5 * iqr) | (df["fare"] > q3 + 1.5 * iqr)]
print(f"\nFare outliers: {len(outliers)} rows ({len(outliers)/len(df):.1%})")

plt.figure(figsize=(6, 4))
sns.boxplot(x=df["fare"])
plt.title("Fare mein Outliers")
plt.tight_layout()
plt.savefig("charts/fare_outliers.png", dpi=150)
plt.close()

# ------------------------------------------------------------------
# STEP 4: Patterns aur trends (sawalon ke jawab)
# ------------------------------------------------------------------
print(f"\nOverall survival rate: {df['survived'].mean():.1%}")

# Q1: Gender vs Survival
print("\nSurvival by sex:\n", df.groupby("sex")["survived"].mean())
plt.figure(figsize=(5, 4))
sns.barplot(x="sex", y="survived", data=df)
plt.title("Gender ke hisaab se Survival Rate")
plt.tight_layout()
plt.savefig("charts/survival_by_sex.png", dpi=150)
plt.close()

# Q2: Class vs Survival
print("\nSurvival by class:\n", df.groupby("pclass")["survived"].mean())
plt.figure(figsize=(5, 4))
sns.barplot(x="pclass", y="survived", data=df)
plt.title("Passenger Class ke hisaab se Survival Rate")
plt.tight_layout()
plt.savefig("charts/survival_by_class.png", dpi=150)
plt.close()

# Q3: Age vs Survival
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="age", hue="survived", bins=30, kde=True)
plt.title("Age Distribution (Survived vs Not Survived)")
plt.tight_layout()
plt.savefig("charts/age_distribution.png", dpi=150)
plt.close()

# Gender + Class dono ek saath
plt.figure(figsize=(7, 4))
sns.barplot(x="pclass", y="survived", hue="sex", data=df)
plt.title("Class aur Gender ka combined asar")
plt.tight_layout()
plt.savefig("charts/class_sex_survival.png", dpi=150)
plt.close()

# Correlation heatmap (sirf numeric columns)
num_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(7, 5))
sns.heatmap(num_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("charts/correlation_heatmap.png", dpi=150)
plt.close()

# ------------------------------------------------------------------
# STEP 5: Hypothesis testing
# ------------------------------------------------------------------
# H0: Gender ka survival se koi relation nahi hai
# H1: Gender aur survival related hain
contingency = pd.crosstab(df["sex"], df["survived"])
chi2, p, _, _ = stats.chi2_contingency(contingency)
print(f"\nChi-square test (sex vs survived): p-value = {p:.2e}")
print("Result:", "H0 reject: relation hai" if p < 0.05 else "H0 reject nahi hua")

# H0: Survived aur not-survived logon ka average fare same hai
fare_yes = df[df["survived"] == 1]["fare"]
fare_no = df[df["survived"] == 0]["fare"]
t, p2 = stats.ttest_ind(fare_yes, fare_no, equal_var=False)
print(f"T-test (fare): p-value = {p2:.2e}")
print("Result:", "Fare ka fark significant hai" if p2 < 0.05 else "Fark significant nahi")

# ------------------------------------------------------------------
# STEP 6: Conclusion (apne shabdon mein README mein likhna)
# ------------------------------------------------------------------
print("\nCharts 'charts/' folder mein save ho gaye. Ab README mein apne findings likho.")


Shape (rows, columns): (891, 15)

First 5 rows:
    survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  

Data types & non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  